### Prediction Pipeline

In [27]:
#importing libraries
import pickle, re
import numpy as np
from tensorflow.keras.models import load_model

from tensorflow.keras.preprocessing import sequence

In [28]:
#loading model and tokenizer

model = load_model("hamlet_word_prediction_model.keras")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 12, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 12, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,658,256 (13.96 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,438,838 (9.30 MB)

In [29]:
with open("hamlet_tokenizer.pkl", "rb") as file_obj:
    tokenizer = pickle.load(file_obj)

word_index = tokenizer.word_index  
reverse_word_index = {value:key for key, value in word_index.items()}
reverse_word_index

{1: 'the',
 2: 'and',
 3: 'to',
 4: 'of',
 5: 'i',
 6: 'you',
 7: 'a',
 8: 'my',
 9: 'it',
 10: 'in',
 11: 'that',
 12: 'ham',
 13: 'is',
 14: 'not',
 15: 'his',
 16: 'this',
 17: 'with',
 18: 'your',
 19: 'but',
 20: 'for',
 21: 'me',
 22: 'lord',
 23: 'as',
 24: 'what',
 25: 'he',
 26: 'be',
 27: 'so',
 28: 'him',
 29: 'haue',
 30: 'king',
 31: 'will',
 32: 'no',
 33: 'our',
 34: 'we',
 35: 'on',
 36: 'are',
 37: 'if',
 38: 'all',
 39: 'then',
 40: 'shall',
 41: 'by',
 42: 'thou',
 43: 'come',
 44: 'or',
 45: 'hamlet',
 46: 'good',
 47: 'do',
 48: 'hor',
 49: 'her',
 50: 'let',
 51: 'now',
 52: 'thy',
 53: 'how',
 54: 'more',
 55: 'they',
 56: 'from',
 57: 'enter',
 58: 'at',
 59: 'was',
 60: 'oh',
 61: 'like',
 62: 'most',
 63: 'there',
 64: 'well',
 65: 'know',
 66: 'selfe',
 67: 'would',
 68: 'them',
 69: 'loue',
 70: 'may',
 71: "'tis",
 72: 'vs',
 73: 'sir',
 74: 'qu',
 75: 'which',
 76: 'did',
 77: 'why',
 78: 'laer',
 79: 'giue',
 80: 'thee',
 81: 'ile',
 82: 'must',
 83: 'hat

In [30]:
#prediction pipeline

sample_text = "How now Horatio? You tremble & look"

sample_text = sample_text.lower()
sample_text = re.sub('[^a-zA-Z ]', " ", sample_text)
sample_text

'how now horatio  you tremble   look'

In [31]:
sample_text_seq = [word_index.get(word, -1)+3 for word in sample_text.split(" ")]
sample_text_seq

[56, 54, 123, 2, 9, 1195, 2, 2, 575]

In [32]:
#padding seq

sample_text_padded_seq = sequence.pad_sequences([sample_text_seq], maxlen=100)
sample_text_padded_seq

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,   56,   54,  123,    2,    9, 1195,    2,    2,
         575]], dtype=int32)

In [33]:
pred = model.predict(sample_text_padded_seq)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step


In [34]:
pred_word= reverse_word_index.get(np.argmax(pred)-3, "oov_word")
pred_word


'a'

'a'